# Task 3

1. Implement word embedding world:

-  either only one of the two: CBOW or Skip-gram,

-  or use packages such as PyTorch, TensorFlow, or others to build a neural network and train the model,

- or implement embedding "look up" and print the most similar words for a given word from the vocabulary.

# Word2vec
- is an algorithm for representing words as vectors (or embeddings), which allows to model the meaning of words and their relationship with each other.

*Embeddings* - is the representation of objects (usually words or phrases) as vectors in higher dimension space. These vectors are numerical representations of objects, and their values often reflect semantic or syntactic properties of objects.

Example: suppose we want to find out how similar the words *"doctor"* and *"physician"* are. After learning the Word2Vec model for English texts, the vectors of these words will be close and the model can calculate similarities, for example, using the cosine distance between vectors.


**Note that for word2vec you can choose to implement only one of two models (CBOW or skip-gram)**

In [8]:
import os # Operating System - for work with files and folders
import re
import spacy
import numpy as np
import plotly.graph_objects as go
from collections import defaultdict

import torch
from torch import nn, optim
from torch.nn.functional import logsigmoid
from tqdm import  tqdm

In [9]:
nlp = spacy.load("en_core_web_sm")

In [10]:
# read the text (eng_Anne_full_abbr.txt)
base_path = os.getcwd()
file_path = os.path.abspath(os.path.join(base_path, "..", "data", "without_chap_and_title", "eng_Anne_full_abbr.txt"))
text = open(file_path, encoding="utf-8").read()

print(text[0:500])

Mistress Rachel Lynde lived just where the Avonlea main road dipped down into a little hollow, fringed with alders and ladies' eardrops and traversed by a brook that had its source away back in the woods of the old Cuthbert place; it was reputed to be an intricate, headlong brook in its earlier course through those woods, with dark secrets of pool and cascade; but by the time it reached Lynde's Hollow it was a quiet, well-conducted little stream, for not even a brook could run past Mistress Rach


In [11]:
def tokenize_paragrah(paragraph, lemmaOn=False):
    """Build list of sentences and list of tokens for each sentence.
        Punctuation is removed
        If lemmatization is applied, all the tokens are lemmatized first
    """
    doc = nlp(paragraph)
    tokenized_text = []
    for sent in doc.sents:
        if lemmaOn:
            tokenized_text.append([token.lemma_ for token in sent if not token.is_punct])
        else:
            tokenized_text.append([token.text.lower() for token in sent if not token.is_punct])
    return tokenized_text


# def build_bag_of_words(tokenized_paragraph):
#     """Builds two dictionaries that map token into index and vice versa
#     """
#     bow = set()
#     for sent in tokenized_paragraph:
#         for token in sent:
#             bow.add(token)
    
#     token2id = {}
#     id2token = {}
#     for i, t in enumerate(bow):                
#         id2token[i+1] = t
#         token2id[t] = i+1
        
#     token2id["unk"] = 0
#     id2token[0] = "unk"
#     return id2token, token2id

def build_vocab(tokenized_paragraph):
    word_freq = defaultdict(int)
    for sent in tokenized_paragraph:
        for token in sent:
            word_freq[token] += 1
    word2idx = {word: idx for idx, word in enumerate(word_freq)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    counts = np.array([word_freq[idx2word[i]] for i in range(len(word2idx))], dtype=np.float32)
    freq_table = counts ** 0.75
    freq_table /= freq_table.sum()
    return idx2word, word2idx, freq_table

def one_hot_encode(idx, vocab_size):
    """Converts token id into one-hot vector
    """
    res = [0] * vocab_size
    res[idx] = 1
    return res

def generate_tokens_pairs(sentence, window=2):
    """Generates list of pairs: (central token, token in window-neiborhood of central token)
    """
    tokens_pairs = []
    n_tokens = len(sentence)
    
    for i, c_token in enumerate(sentence):
        for idx in range(max(0, i - window), i):
            tokens_pairs.append((c_token, sentence[idx]))
        for idx in range(i+1, min(n_tokens, i + window + 1)):
            tokens_pairs.append((c_token, sentence[idx]))
    return tokens_pairs
        
def get_negative_samples(vocab_size, neg_sample_size, freq_table):
    return np.random.choice(np.arange(vocab_size), size=neg_sample_size, p=freq_table)

def build_training_set(tokens_pairs, token2id, freq_table, neg_sample_size=5):
    X = []
    y = []
    neg_y = []
    for x_token, y_token in tokens_pairs:
        X.append(one_hot_encode(token2id[x_token], len(token2id)))
        y.append(one_hot_encode(token2id[y_token], len(token2id)))
        neg_ids = get_negative_samples(len(token2id), neg_sample_size, freq_table)
        # for i in neg_ids:
        #     print(i, tokens_pairs[i][1])
        #print(neg_ids)
        neg_y.append([one_hot_encode(token2id[tokens_pairs[i][1]], len(token2id)) for i in neg_ids])
    return np.array(X), np.array(y), np.array(neg_y)

In [ ]:
tokenized_text = tokenize_paragrah(text)
#id2token, token2id = build_bag_of_words(tokenized_text)
id2token, token2id, freq_table = build_vocab(tokenized_text)
print(token2id)

vocab_size = len(id2token)
print(vocab_size)

print(tokenized_text[0])

#tokens_pairs = generate_tokens_pairs(tokenized_text[0])
tokens_pairs = []
for sentence in tokenized_text:
    tokens_pairs.extend(generate_tokens_pairs(sentence))
train_X, train_y, train_neg_y = build_training_set(tokens_pairs, token2id, freq_table)

print(train_X.shape)

{'mistress': 0, 'rachel': 1, 'lynde': 2, 'lived': 3, 'just': 4, 'where': 5, 'the': 6, 'avonlea': 7, 'main': 8, 'road': 9, 'dipped': 10, 'down': 11, 'into': 12, 'a': 13, 'little': 14, 'hollow': 15, 'fringed': 16, 'with': 17, 'alders': 18, 'and': 19, 'ladies': 20, 'eardrops': 21, 'traversed': 22, 'by': 23, 'brook': 24, 'that': 25, 'had': 26, 'its': 27, 'source': 28, 'away': 29, 'back': 30, 'in': 31, 'woods': 32, 'of': 33, 'old': 34, 'cuthbert': 35, 'place': 36, 'it': 37, 'was': 38, 'reputed': 39, 'to': 40, 'be': 41, 'an': 42, 'intricate': 43, 'headlong': 44, 'earlier': 45, 'course': 46, 'through': 47, 'those': 48, 'dark': 49, 'secrets': 50, 'pool': 51, 'cascade': 52, 'but': 53, 'time': 54, 'reached': 55, "'s": 56, 'quiet': 57, 'well': 58, 'conducted': 59, 'stream': 60, 'for': 61, 'not': 62, 'even': 63, 'could': 64, 'run': 65, 'past': 66, 'door': 67, 'without': 68, 'due': 69, 'regard': 70, 'decency': 71, 'decorum': 72, 'probably': 73, 'conscious': 74, 'sitting': 75, 'at': 76, 'her': 77, '

In [ ]:
print("vocabulary position:", tokens_pairs[0], token2id[tokens_pairs[0][0]], token2id[tokens_pairs[0][1]])
(train_X[0], train_y[0], train_neg_y[0])
print("one-hot indices:", train_X[0].argmax(), train_y[0].argmax())
for i in train_neg_y[0]:
    print(id2token[i.argmax()])

NameError: name 'tokens_pairs' is not defined

In [ ]:
class CBOW(nn.Module):
    """
    Continuous bag of words
    """

    def __init__(self, vocab_size, n_embeddings) -> None:
        super().__init__()

        self.vocab_size = vocab_size
        self.vector_dim = n_embeddings
        self.W1 = nn.Parameter(data=torch.randn(self.vocab_size, self.vector_dim), requires_grad=True) # Word Vectors
        self.W2 = nn.Parameter(data=torch.randn(self.vector_dim, self.vocab_size), requires_grad=True)

    def forward(self, X) -> torch.tensor:
        X = X @ self.W1
        X = X @ self.W2
        return X

In [ ]:
class SGNS(nn.Module):
    """
    Skip-gram negative sampling
    """

    def __init__(self, vocab_size, embedding_dim):
        super(SGNS, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim

        # Initialize embedding matrices as parameters
        self.input_weights = nn.Parameter(torch.randn(vocab_size, embedding_dim) * 0.01, requires_grad=True)
        self.output_weights = nn.Parameter(torch.randn(vocab_size, embedding_dim) * 0.01, requires_grad=True)

    def forward(self, center_onehot, pos_onehot, neg_onehots):
        # Convert one-hot to embeddings via matrix multiply
        v_c = center_onehot @ self.input_weights  # (batch_size, embedding_dim)
        u_o = pos_onehot @ self.output_weights    # (batch_size, embedding_dim)
        u_k = neg_onehots @ self.output_weights   # (batch_size, neg_samples, embedding_dim)

        # Positive loss: log σ(uᵀv)
        pos_score = torch.sum(v_c * u_o, dim=1)
        pos_loss = logsigmoid(pos_score)

        # Negative loss: Σ log σ(-uₖᵀv)
        neg_score = torch.bmm(u_k.neg(), v_c.unsqueeze(2)).squeeze()
        neg_loss = logsigmoid(neg_score).sum(1)

        loss = -(pos_loss + neg_loss).mean()
        return loss

    def get_embeddings(self):
        return self.input_weights.data

# def get_negative_samples(batch_size, vocab_size, neg_sample_size, freq_table):
#     return np.random.choice(np.arange(vocab_size), size=(batch_size, neg_sample_size), p=freq_table) 

In [ ]:
x = torch.tensor(train_X, dtype=torch.float)
print(x.size())
y = torch.tensor(train_y, dtype=torch.float)
print(y.size())
y_neg = torch.tensor(train_neg_y, dtype=torch.float)
print(y_neg.size())

In [ ]:
eps = 0.1
vocab_size = len(id2token)
n_embeddings = 10

cbow_model = CBOW(vocab_size, n_embeddings)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=cbow_model.parameters(), lr=eps)

error = []
for epoch in tqdm(range(100)):
    pred = cbow_model(x)
    train_loss = loss_fn(pred, y)
    error.append(train_loss.item())
    
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    
fig = go.Figure(data=[go.Scatter(y=error)])
fig.show()

In [ ]:
eps = 0.1
vocab_size = len(id2token)
n_embeddings = 10
error = []

sgns_model = SGNS(vocab_size, n_embeddings)
optimizer = torch.optim.Adam(params=sgns_model.parameters(), lr=eps)

for epoch in tqdm(range(100)):
    loss = sgns_model(x, y, y_neg)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    #print(f"Epoch {epoch+1}, Loss: {loss}")
    error.append(loss.item())

fig = go.Figure(data=[go.Scatter(y=error)])
fig.show()

In [ ]:
params = list(sgns_model.parameters())
word_vectors = params[0].detach()
print(word_vectors.size())

token2vec_sgns = {t: word_vectors[idx] for t, idx in token2id.items()}
token2vec_sgns

In [ ]:
params = list(cbow_model.parameters())
word_vectors = params[0].detach()
print(word_vectors.size())

token2vec_cbow = {t: word_vectors[idx] for t, idx in token2id.items()}
token2vec_cbow

In [ ]:
def cosine_similarity(v1, v2):
    return (v1 @ v2) / (torch.norm(v1) * torch.norm(v2))

def most_similar(word, word_dict, top_k=5):
    if word not in word_dict:
        raise ValueError(f"{word} not found in the word dictionary.")

    query_vector = word_dict[word]

    # Calculate cosine similarity with all other words in the dictionary
    similarities = {}
    for other_word, other_vector in word_dict.items():
        if other_word != word:
            similarity = cosine_similarity(query_vector, other_vector)
            similarities[other_word] = similarity

    # Sort the words by similarity in descending order
    sorted_similarities = sorted(similarities.items(), key=lambda x: x[1], reverse=True)

    # Get the top-k most similar words
    top_similar_words = sorted_similarities[:top_k]

    return top_similar_words

In [ ]:
most_similar("learning", token2vec_cbow)

In [ ]:
most_similar("learning", token2vec_sgns)

In [ ]:
## try different embedding sizes and windows and make conclusions